<a href="https://colab.research.google.com/github/joyhwp/lg_multimodal_cej_analysis/blob/main/04_cluster_insight_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install -y fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl

# 폰트 경로 확인
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

# 폰트 등록
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 깨짐 방지

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 3s (3,692 kB/s)
Selecting previously unselected package fonts-nanum.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from mpl_toolkits.mplot3d import Axes3D

In [3]:
import pandas as pd
df = pd.read_csv('ohouse_image_subcluster_result_final_visual_labeled.csv')
df.columns

Index(['text_cluster_df_index', 'product_id', 'product_name', 'product_url',
       'styling_id', 'styling_url', 'thumbnail_url', 'local_image_path',
       'view_count', 'like_count', 'bookmark_count', 'author', 'caption',
       'raw_text', 'collected_at', 'image_path', 'image_filename',
       'image_stem', 'image_ext', 'product_id_from_image', 'image_seq',
       'file_size_bytes', 'ext_priority', 'caption_clean', 'raw_text_clean',
       'text_for_analysis', 'text_length', 'text_for_clip',
       'text_for_clip_before_clean', 'text_quality',
       'qwen_image_text_similarity', 'similarity_status', 'analysis_row_id',
       'text_cluster', 'text_cluster_prob', 'umap_x', 'umap_y',
       'text_cluster_label', 'cej_stage', 'image_subcluster',
       'image_subcluster_global', 'image_cluster_k',
       'image_cluster_silhouette', 'image_k_decision_type',
       'image_subcluster_size', 'image_subcluster_is_small',
       'image_subcluster_visual_label', 'image_subcluster_review_note'

In [5]:
"""
오늘의집 텍스트 군집 × 이미지 서브군집 뷰어
Colab에서 실행하세요.

사용법:
1. df 변수명을 실제 데이터프레임 변수명으로 바꾸세요
2. 셀 단위로 실행하면 T0I0, T0I1... 순서로 이미지 그리드가 출력됩니다
"""

# ── 설정 ──────────────────────────────────────────────────────────────────────
DF_VAR         = "df"           # 실제 데이터프레임 변수명
IMAGE_URL_COL  = "thumbnail_url"           # 이미지 URL 컬럼
TEXT_CLUSTER   = "text_cluster"            # 텍스트 군집 컬럼
IMAGE_CLUSTER  = "image_subcluster_global" # 이미지 군집 컬럼 (global 단위)
LABEL_COL      = "image_subcluster_visual_label"  # 이미지 군집 라벨
NOTE_COL       = "image_subcluster_review_note"   # 리뷰 노트
TEXT_LABEL_COL = "text_cluster_label"             # 텍스트 군집 라벨

MAX_IMAGES     = 30  # 조합당 최대 표시 이미지 수
IMG_SIZE       = 160  # 픽셀 (그리드 셀 크기)
# ─────────────────────────────────────────────────────────────────────────────

import math
import textwrap
from IPython.display import display, HTML


def make_grid_html(subset, combo_label, img_label, note, n_total):
    """한 T×I 조합의 이미지 그리드 HTML 반환"""
    urls = subset[IMAGE_URL_COL].dropna().tolist()[:MAX_IMAGES]

    img_tags = ""
    for url in urls:
        img_tags += f"""
        <div style="display:inline-block; margin:4px; vertical-align:top;">
          <img src="{url}"
               style="width:{IMG_SIZE}px; height:{IMG_SIZE}px;
                      object-fit:cover; border-radius:6px;
                      border:1px solid #ddd;"
               onerror="this.style.display='none'">
        </div>"""

    note_text = textwrap.shorten(str(note) if note else "", width=120, placeholder="...")
    label_text = str(img_label) if img_label else "라벨 없음"

    html = f"""
    <div style="margin:20px 0; padding:14px; background:#f9f9f9;
                border-left:4px solid #4A90D9; border-radius:8px;">
      <div style="font-size:17px; font-weight:bold; color:#222; margin-bottom:4px;">
        {combo_label}
      </div>
      <div style="font-size:13px; color:#555; margin-bottom:2px;">
        🏷 이미지 라벨: <b>{label_text}</b>
      </div>
      <div style="font-size:12px; color:#888; margin-bottom:10px;">
        📝 {note_text} &nbsp;|&nbsp; 전체 {n_total}장 중 최대 {MAX_IMAGES}장 표시
      </div>
      <div>{img_tags}</div>
    </div>"""
    return html


def view_clusters(df,
                  text_clusters=None,
                  image_clusters=None,
                  min_size=3):
    """
    Parameters
    ----------
    df            : 데이터프레임
    text_clusters : 볼 텍스트 군집 번호 리스트. None이면 전체
    image_clusters: 볼 이미지 군집 번호 리스트. None이면 전체
    min_size      : 이 이상 이미지가 있는 조합만 표시
    """
    t_list = sorted(df[TEXT_CLUSTER].unique()) if text_clusters is None else text_clusters
    i_list = sorted(df[IMAGE_CLUSTER].unique()) if image_clusters is None else image_clusters

    for t in t_list:
        t_df = df[df[TEXT_CLUSTER] == t]
        t_label = t_df[TEXT_LABEL_COL].dropna().iloc[0] if TEXT_LABEL_COL in t_df.columns and len(t_df[TEXT_LABEL_COL].dropna()) > 0 else ""

        # 텍스트 군집 헤더
        display(HTML(f"""
        <div style="margin-top:30px; padding:10px 16px;
                    background:#1a1a2e; color:#fff;
                    border-radius:8px; font-size:18px; font-weight:bold;">
          📂 T{t} &nbsp;|&nbsp; {t_label}
          <span style="font-size:13px; font-weight:normal; color:#aaa;">
            (총 {len(t_df)}행)
          </span>
        </div>"""))

        found = False
        for i in i_list:
            subset = t_df[t_df[IMAGE_CLUSTER] == i]
            if len(subset) < min_size:
                continue
            found = True

            # 대표 라벨/노트
            def safe_mode(series):
                m = series.dropna()
                return m.mode().iloc[0] if len(m.mode()) > 0 else ""

            img_label = safe_mode(subset[LABEL_COL]) if LABEL_COL in subset.columns else ""
            note      = safe_mode(subset[NOTE_COL])  if NOTE_COL  in subset.columns else ""

            combo = f"T{t} × I{i}"
            html = make_grid_html(subset, combo, img_label, note, len(subset))
            display(HTML(html))

        if not found:
            display(HTML("<div style='color:#aaa; margin:8px 16px;'>해당 조합 없음 (min_size 미달)</div>"))


# ── 실행 예시 ─────────────────────────────────────────────────────────────────

# 전체 보기
view_clusters(df)

# 특정 텍스트 군집만 (예: LG브랜드=2, 구매후기=5, 디자인만족=4)
# view_clusters(df, text_clusters=[2, 4, 5])

# 특정 텍스트 × 특정 이미지 조합만
# view_clusters(df, text_clusters=[2, 4, 5], image_clusters=[0, 1, 2, 3])

# ── 여기서 실행 ───────────────────────────────────────────────────────────────
view_clusters(df, text_clusters=[2, 4, 5])  # LG 관심 높은 군집 우선

In [6]:
import math
import textwrap
from IPython.display import display, HTML

# ── 설정
IMAGE_URL_COL  = "thumbnail_url"
TEXT_CLUSTER   = "text_cluster"
IMAGE_CLUSTER  = "image_subcluster_global"
LABEL_COL      = "image_subcluster_visual_label"
NOTE_COL       = "image_subcluster_review_note"
TEXT_LABEL_COL = "text_cluster_label"
SIM_COL        = "qwen_image_text_similarity"
MAX_IMAGES     = 30
IMG_SIZE       = 160

def safe_mode(series):
    m = series.dropna()
    return m.mode().iloc[0] if len(m.mode()) > 0 else ""

def make_grid_html(subset, combo_label, img_label, note, n_total):
    # 유사도 높은 순 정렬
    sorted_sub = subset.sort_values(SIM_COL, ascending=False)
    urls = sorted_sub[IMAGE_URL_COL].dropna().tolist()[:MAX_IMAGES]

    img_tags = ""
    for url in urls:
        img_tags += f"""
        <div style="display:inline-block; margin:4px; vertical-align:top;">
          <img src="{url}"
               style="width:{IMG_SIZE}px; height:{IMG_SIZE}px;
                      object-fit:cover; border-radius:6px;
                      border:1px solid #ddd;"
               onerror="this.style.display='none'">
        </div>"""

    note_text  = textwrap.shorten(str(note) if note else "", width=120, placeholder="...")
    label_text = str(img_label) if img_label else "라벨 없음"
    sim_vals   = subset[SIM_COL].dropna()
    sim_info   = f"avg={sim_vals.mean():.3f} / max={sim_vals.max():.3f}"

    return f"""
    <div style="margin:20px 0; padding:14px; background:#f9f9f9;
                border-left:4px solid #4A90D9; border-radius:8px;">
      <div style="font-size:17px; font-weight:bold; color:#222; margin-bottom:4px;">
        {combo_label}
      </div>
      <div style="font-size:13px; color:#555; margin-bottom:2px;">
        🏷 {label_text} &nbsp;|&nbsp; 📊 유사도 {sim_info} &nbsp;|&nbsp; {n_total}장 중 {min(MAX_IMAGES, n_total)}장
      </div>
      <div style="font-size:12px; color:#888; margin-bottom:10px;">
        📝 {note_text}
      </div>
      <div>{img_tags}</div>
    </div>"""

def view_clusters(df, text_clusters=None, min_size=3):
    # text_cluster 기준으로 순회
    t_list = sorted(df[TEXT_CLUSTER].unique()) if text_clusters is None else text_clusters

    for t in t_list:
        t_df     = t_df = df[df[TEXT_CLUSTER] == t]
        t_label  = safe_mode(t_df[TEXT_LABEL_COL]) if TEXT_LABEL_COL in t_df.columns else ""

        display(HTML(f"""
        <div style="margin-top:30px; padding:10px 16px;
                    background:#1a1a2e; color:#fff;
                    border-radius:8px; font-size:18px; font-weight:bold;">
          📂 TC {t} &nbsp;|&nbsp; {t_label}
          <span style="font-size:13px; font-weight:normal; color:#aaa;">
            ({len(t_df)}행)
          </span>
        </div>"""))

        # image_subcluster_global은 "T0_I0" 형식 — 해당 TC의 것만
        i_groups = sorted(t_df[IMAGE_CLUSTER].dropna().unique())
        found    = False

        for ig in i_groups:
            subset = t_df[t_df[IMAGE_CLUSTER] == ig]
            if len(subset) < min_size:
                continue
            found = True

            img_label = safe_mode(subset[LABEL_COL]) if LABEL_COL in subset.columns else ""
            note      = safe_mode(subset[NOTE_COL])  if NOTE_COL  in subset.columns else ""

            html = make_grid_html(subset, ig, img_label, note, len(subset))
            display(HTML(html))

        if not found:
            display(HTML("<div style='color:#aaa; margin:8px 16px;'>해당 조합 없음</div>"))

# ── 실행
view_clusters(df)

# 특정 텍스트 군집만 보려면
# view_clusters(df, text_clusters=[0, 1, 3])

In [7]:
import re

# LG 관련 키워드 패턴
lg_patterns = {
    "LG 브랜드 언급":   r"lg|엘지|엘쥐",
    "LG 긍정":         r"역시\s*엘?지|엘?지\s*최고|엘?지\s*짱|가전은\s*엘?지|역시\s*lg",
    "임직원 할인":      r"임직원|할인",
    "기사님 친절":      r"기사님|친절|설치\s*기사",
    "가격·가성비":      r"가격|가성비|저렴|싸게|할인",
    "공간·인테리어":    r"인테리어|어울|공간|색감|톤|컬러|주방|냉툭튀|키친핏",
    "성능·기능":        r"성능|기능|조용|소음|냉동|냉장|용량|넓",
}

target = df[
    (df["text_cluster"] == 5) &
    (df["image_subcluster_global"].isin(["T5_I0", "T5_I2"]))
].copy()

target["cleaned"] = target["raw_text_clean"].apply(lambda x: re.sub(
    r'\S+\s*팔로우\s*조회수\s*\d*\s*', '', str(x)
).lower())

rows = []
for pattern_name, pattern in lg_patterns.items():
    for grp in ["T5_I0", "T5_I2"]:
        sub = target[target["image_subcluster_global"] == grp]
        n_total = len(sub)
        n_match = sub["cleaned"].str.contains(pattern, regex=True, na=False).sum()
        rows.append({
            "키워드":   pattern_name,
            "군집":     grp,
            "n":        n_total,
            "언급수":   n_match,
            "언급률":   f"{n_match/n_total*100:.1f}%",
        })

result_df = pd.DataFrame(rows)
pivot = result_df.pivot(index="키워드", columns="군집", values="언급률")
display(pivot)

# 언급된 실제 텍스트 확인
print("\n=== T5_I0 공간·인테리어 언급 샘플 ===")
mask = (target["image_subcluster_global"]=="T5_I0") & \
       (target["cleaned"].str.contains(lg_patterns["공간·인테리어"], regex=True, na=False))
for t in target[mask]["cleaned"].head(3):
    print(f"  · {t[:150]}")

print("\n=== T5_I2 공간·인테리어 언급 샘플 ===")
mask = (target["image_subcluster_global"]=="T5_I2") & \
       (target["cleaned"].str.contains(lg_patterns["공간·인테리어"], regex=True, na=False))
for t in target[mask]["cleaned"].head(3):
    print(f"  · {t[:150]}")

군집,T5_I0,T5_I2
키워드,,
LG 긍정,6.8%,0.0%
LG 브랜드 언급,34.1%,18.2%
가격·가성비,22.7%,4.5%
공간·인테리어,20.5%,45.5%
기사님 친절,47.7%,9.1%
성능·기능,95.5%,90.9%
임직원 할인,4.5%,0.0%



=== T5_I0 공간·인테리어 언급 샘플 ===
  · 이렇게 저렴한 가격에 엘지 냉장고라니 컬러가 은은하고 어디에 두어도 잘 어울려서 인테리어 효과가 좋아요 성능은 말하나 마나 좋고 전체적으로 깔끔하고 고급스러워 만족스러워요.
  · 3 다음은 냉장고입니다. 저희는 가전을 lg로 맞췄는데요. 임직원 할인 활용해서 저렴하게 했어요. 처음 냉장고 들이고 놀랬던 게 집 옵션이 lg로 되어있어서 그런지 냉장고 컬러와 펜트리 컬러가 맞춘 것처럼 찰떡이더라구요. 덕분에 시선이 끊기지 않고 이어져서 더 공간이 
  · 냉장고 용량이 커서 너무 좋아요 툭튀어 나오긴 하지만 정리하기 좋고 저희집에 잘 어울리는 화이트라 더 마음에 듭니다 빠른배송 감사합니다

=== T5_I2 공간·인테리어 언급 샘플 ===
  · 2 처음엔 냉툭튀가 걱정되어서 키친핏으로 사야하나 고민했는데요. 그래도 저장 공간이 중요하겠다 싶어서 일반 모델로 구매했습니다. 다행히 lg 냉장고는 도어 옆면도 베이지 톤으로 되어있어서 생각보다 냉툭튀 티 거의 안나요! 침실
  · 1 깔끔히 화이트톤에 맞춘 주방에 잘 어울리는 오브제 베이지 컬러! 빌트인이 아니라 정돈되지 않는 게 조금 걸리긴 하지만 저는 오히려 깊이가 있어 냉장 공간이 넓은 탓에 선택한 냉장고예요! 아무래도 요리를 많이 해야 하는 상황이다 보니 냉장고도 커야 좋더라고요.
  · 1 #살림살이 #콘수프기록집 원래 얼음정수기를 쓰다가 이사오면서 위약금 물고 얼음정수기 처분하고 퓨리케어를 쓰고 있어요!! 인테리어 공사할때 퓨리케어를 설치할꺼면 배관공사를 따로 해여한다고 하더라구요! 안 했으면 너무 후회했을꺼같아요! 처음엔 얼음얼리는게 너무 귀찮았는


In [8]:
def analyze_text_cluster(df, tc, text_col="raw_text_clean", top_n=20):
    import re

    def clean_text(text):
        text = str(text)
        # 오늘의집 헤더 노이즈 제거 (닉네임 팔로우 조회수 패턴)
        text = re.sub(r'^.*?팔로우\s*조회수\s*\d*\s*', '', text)
        text = re.sub(r'\S+\s*팔로우\s*조회수\s*\d*\s*', '', text)
        text = re.sub(r'팔로우|조회수', '', text)
        text = re.sub(r'\d+:\d+', '', text)  # 시간 형식
        text = re.sub(r'http\S+', ' ', text)
        text = re.sub(r'[^\w\s가-힣]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    STOPWORDS = [
        "냉장고", "이", "그", "저", "것", "수", "있", "없", "하", "되",
        "않", "나", "우리", "제", "좀", "더", "도", "를", "을",
        "가", "은", "는", "에", "의", "로", "으로", "만",
        "정말", "너무", "진짜", "완전", "그냥", "그리고", "그래서",
        "근데", "하지만", "해요", "해서", "했어", "있어", "같아",
        "같은", "같이", "거", "좋은", "좋아", "좋고", "좋게",
        "팔로우", "조회수", "북마크",  # ← 추가
    ]

    subset = df[df["text_cluster"] == tc].copy()
    subset["cleaned"] = subset[text_col].apply(clean_text)

    # 너무 짧아진 텍스트 제거
    subset = subset[subset["cleaned"].str.len() >= 10]
    texts = subset["cleaned"].tolist()

    print(f"{'='*70}")
    print(f"TC {tc} | {df[df['text_cluster']==tc]['text_cluster_label'].dropna().iloc[0] if 'text_cluster_label' in df.columns else ''}")
    print(f"유효 텍스트 {len(texts)}개")
    print(f"{'='*70}\n")

    # 1. c-TF-IDF 키워드
    print("── [1] c-TF-IDF 핵심 키워드 ──")
    try:
        vec   = CountVectorizer(
            token_pattern=r"(?u)\b[가-힣A-Za-z]{2,}\b",
            ngram_range=(1, 2),
            stop_words=STOPWORDS,
            max_features=5000
        )
        X     = vec.fit_transform(texts)
        tfidf = TfidfTransformer(norm=None, use_idf=True, smooth_idf=True)
        Xt    = tfidf.fit_transform(X)
        terms = vec.get_feature_names_out()
        scores = Xt.toarray().sum(axis=0)
        top_idx = scores.argsort()[::-1][:top_n]
        for rank, i in enumerate(top_idx, 1):
            print(f"  {rank:2d}. {terms[i]:<15} ({scores[i]:.1f})")
    except Exception as e:
        print(f"  오류: {e}")

    print()

    # 2. 이미지 서브군집별 키워드 비교
    print("── [2] 이미지 서브군집별 주요 키워드 ──")
    for ig in sorted(subset["image_subcluster_global"].dropna().unique()):
        ig_texts = subset[subset["image_subcluster_global"] == ig]["cleaned"].tolist()
        if len(ig_texts) < 3:
            continue
        try:
            vec_ig = CountVectorizer(
                token_pattern=r"(?u)\b[가-힣A-Za-z]{2,}\b",
                ngram_range=(1, 2),
                stop_words=STOPWORDS,
                max_features=2000
            )
            X_ig     = vec_ig.fit_transform(ig_texts)
            t_ig     = TfidfTransformer(norm=None).fit_transform(X_ig)
            terms_ig = vec_ig.get_feature_names_out()
            sc_ig    = t_ig.toarray().sum(axis=0)
            top_ig   = sc_ig.argsort()[::-1][:10]
            kws      = ", ".join(terms_ig[top_ig])
            label    = subset[subset["image_subcluster_global"]==ig]["image_subcluster_visual_label"].dropna().mode()
            label    = label.iloc[0] if len(label) else ""
            print(f"  {ig} ({len(ig_texts)}건) [{label}]")
            print(f"    → {kws}")
        except Exception as e:
            print(f"  {ig} 오류: {e}")

    print()

    # 3. 정제된 대표 텍스트
    print("── [3] 대표 텍스트 샘플 ──")
    samples = subset[
        (subset["cleaned"].str.len() >= 30) &
        (subset["cleaned"].str.len() <= 300)
    ]["cleaned"].tolist()[:5]
    for i, s in enumerate(samples, 1):
        print(f"  [{i}] {s[:200]}")
        print()

# 실행
analyze_text_cluster(df, tc=5)

TC 5 | 냉장고 구매·배송·용량 만족
유효 텍스트 90개

── [1] c-TF-IDF 핵심 키워드 ──
  오류: name 'CountVectorizer' is not defined

── [2] 이미지 서브군집별 주요 키워드 ──
  T5_I0 오류: name 'CountVectorizer' is not defined
  T5_I1 오류: name 'CountVectorizer' is not defined
  T5_I2 오류: name 'CountVectorizer' is not defined

── [3] 대표 텍스트 샘플 ──
  [1] 냉장고 자체는 깔끔하니 좋습니다 성능도 좋구요 가격도 만족스럽습니다 냉동실도 넓어서 사용하는데 편하네요 냉장고를 미리 주문하길 잘 한 듯 합니다

  [2] 싸이즈 넉넉하고 2등급이예요 엄청 조용하구요 설치하러오신 LG기사님두분 너무고생하셨습니다 엄청 친절하셨어요 기존냉장고도 치워주셨구요 오늘의집에서 제일 싸게 구매했습니다 다알아보니 냉장고는 역쉬 엘지라는 ᆢ잘구매했어요

  [3] 이렇게 저렴한 가격에 엘지 냉장고라니 컬러가 은은하고 어디에 두어도 잘 어울려서 인테리어 효과가 좋아요 성능은 말하나 마나 좋고 전체적으로 깔끔하고 고급스러워 만족스러워요

  [4] 3 다음은 냉장고입니다 저희는 가전을 LG로 맞췄는데요 임직원 할인 활용해서 저렴하게 했어요 처음 냉장고 들이고 놀랬던 게 집 옵션이 LG로 되어있어서 그런지 냉장고 컬러와 펜트리 컬러가 맞춘 것처럼 찰떡이더라구요 덕분에 시선이 끊기지 않고 이어져서 더 공간이 넓어 보이는 것 같아요

  [5] 2 처음엔 냉툭튀가 걱정되어서 키친핏으로 사야하나 고민했는데요 그래도 저장 공간이 중요하겠다 싶어서 일반 모델로 구매했습니다 다행히 LG 냉장고는 도어 옆면도 베이지 톤으로 되어있어서 생각보다 냉툭튀 티 거의 안나요 침실



In [9]:
import re

신혼_patterns = {
    "신혼 관련":    r"신혼|신랑|신부|결혼|혼수",
    "인테리어 관련": r"인테리어|꾸미|스타일링|시공|몰딩|타일",
    "주방 관련":    r"주방|부엌|키친|씽크|렌지",
    "LG 언급":     r"lg|엘지|오브제",
    "해시태그 중심": r"#|맞팔|콘수프|오늘의우리집",
}

subset = df[df["text_cluster"] == 8].copy()
subset["cleaned"] = subset["raw_text_clean"].apply(lambda x: re.sub(
    r'\S+\s*팔로우\s*조회수\s*\d*\s*', '', str(x)
).lower())

n_total = len(subset)
print(f"TC8 전체: {n_total}건\n")

rows = []
for name, pattern in 신혼_patterns.items():
    n_match = subset["cleaned"].str.contains(pattern, regex=True, na=False).sum()
    rows.append({
        "키워드 유형": name,
        "언급 수":    n_match,
        "비율":       f"{n_match/n_total*100:.1f}%"
    })

result = pd.DataFrame(rows)
display(result)

# 이미지 서브군집별 비교
print("\n=== 이미지 서브군집별 신혼 언급 비율 ===")
for ig in sorted(subset["image_subcluster_global"].dropna().unique()):
    ig_sub = subset[subset["image_subcluster_global"] == ig]
    n_ig = len(ig_sub)
    n_신혼 = ig_sub["cleaned"].str.contains(r"신혼|신랑|신부|결혼|혼수", regex=True, na=False).sum()
    n_인테리어 = ig_sub["cleaned"].str.contains(r"인테리어|꾸미|스타일링|시공", regex=True, na=False).sum()
    print(f"  {ig} ({n_ig}건) | 신혼={n_신혼/n_ig*100:.1f}% | 인테리어={n_인테리어/n_ig*100:.1f}%")

TC8 전체: 72건



,키워드 유형,언급 수,비율
0,신혼 관련,34,47.2%
1,인테리어 관련,56,77.8%
2,주방 관련,61,84.7%
3,LG 언급,1,1.4%
4,해시태그 중심,61,84.7%



=== 이미지 서브군집별 신혼 언급 비율 ===
  T8_I0 (24건) | 신혼=45.8% | 인테리어=70.8%
  T8_I1 (48건) | 신혼=47.9% | 인테리어=75.0%


In [10]:
"""
CEJ 설치 단계 텍스트 리뷰 특징 분석
Colab에서 실행하세요.
"""

from IPython.display import display, HTML
import pandas as pd
from collections import Counter
import re

# ── 설정 ──────────────────────────────────────────────────────────────────────
CEJ_COL        = "cej_stage"
TEXT_COL       = "raw_text_clean"       # 없으면 "raw_text" 또는 "caption_clean"
TEXT_CLUSTER   = "text_cluster"
TEXT_LABEL_COL = "text_cluster_label"
IMAGE_CLUSTER  = "image_subcluster_global"
IMG_LABEL_COL  = "image_subcluster_visual_label"

# 설치 단계에 해당하는 CEJ 값 (데이터에 맞게 조정)
INSTALL_STAGES = ["설치/사용/관리", "설치/사용", "구매/설치/사용", "사용"]

N_SAMPLE    = 10   # 군집당 샘플 리뷰 수
TOP_WORDS   = 30   # 상위 키워드 수
# ─────────────────────────────────────────────────────────────────────────────


def show_header(text, color="#2471A3"):
    display(HTML(f"""
    <div style="margin-top:30px; padding:10px 18px; background:{color};
                color:#fff; border-radius:8px; font-size:18px; font-weight:bold;">
        {text}
    </div>"""))


def show_subheader(text, color="#D6EAF8"):
    display(HTML(f"""
    <div style="margin:14px 0 6px 0; padding:8px 14px; background:{color};
                border-radius:6px; font-size:14px; font-weight:bold; color:#1a1a2e;">
        {text}
    </div>"""))


def get_top_words(texts, top_n=TOP_WORDS):
    """간단한 어절 빈도 분석 (형태소 분석기 없이)"""
    stopwords = {"있어요", "있는", "있고", "있어", "해요", "하고", "하는", "이고",
                 "그리고", "그냥", "정말", "너무", "진짜", "같아요", "같은", "같고",
                 "했어요", "했고", "해서", "하면", "이라", "으로", "에서", "에게",
                 "것도", "것이", "것은", "것을", "거라", "거예요", "거고", "이에요",
                 "이랑", "이나", "이게", "이건", "이런", "이번", "ㅎㅎ", "ㅋㅋ", "^^"}
    words = []
    for text in texts.dropna():
        tokens = re.findall(r'[가-힣]{2,}', str(text))
        words.extend([w for w in tokens if w not in stopwords])
    return Counter(words).most_common(top_n)


def show_keyword_bar(word_counts, title="상위 키워드"):
    """키워드 빈도 바 차트 HTML"""
    if not word_counts:
        return
    max_count = word_counts[0][1]
    bars = ""
    for word, count in word_counts:
        pct = count / max_count * 100
        bars += f"""
        <div style="display:flex; align-items:center; margin:3px 0;">
          <div style="width:80px; font-size:12px; text-align:right;
                      padding-right:8px; color:#333;">{word}</div>
          <div style="background:#2980B9; height:16px; width:{pct:.0f}%;
                      border-radius:3px; min-width:4px;"></div>
          <div style="font-size:11px; color:#666; margin-left:6px;">{count}</div>
        </div>"""
    display(HTML(f"""
    <div style="margin:10px 0; padding:12px; background:#f8f8f8; border-radius:6px;">
      <div style="font-size:13px; font-weight:bold; margin-bottom:8px;">{title}</div>
      {bars}
    </div>"""))


def show_sample_reviews(subset, n=N_SAMPLE):
    """샘플 리뷰 텍스트 출력"""
    samples = subset[TEXT_COL].dropna().sample(min(n, len(subset)), random_state=42).tolist()
    items = "".join([f"<li style='margin:4px 0; font-size:13px; color:#333;'>{s[:200]}</li>"
                     for s in samples])
    display(HTML(f"<ul style='margin:0; padding-left:20px;'>{items}</ul>"))


def analyze_install_stage(df):
    install_df = df[df[CEJ_COL].isin(INSTALL_STAGES)]

    show_header(f"🔧 설치 단계 전체 개요 ({len(install_df)}행)")

    # 1. CEJ 세부 단계별 분포
    stage_counts = install_df[CEJ_COL].value_counts()
    display(stage_counts.to_frame("건수"))

    # 2. 텍스트 군집별 분석
    show_header("📂 텍스트 군집별 분석", color="#1A5276")

    for t in sorted(install_df[TEXT_CLUSTER].unique()):
        t_df = install_df[install_df[TEXT_CLUSTER] == t]
        t_label = t_df[TEXT_LABEL_COL].dropna().iloc[0] \
                  if len(t_df[TEXT_LABEL_COL].dropna()) > 0 else ""

        show_subheader(f"T{t} | {t_label} ({len(t_df)}건)")

        # 키워드
        words = get_top_words(t_df[TEXT_COL], top_n=20)
        show_keyword_bar(words, title=f"T{t} 상위 키워드")

        # 샘플 리뷰
        display(HTML("<div style='font-size:13px; font-weight:bold; margin:8px 0 4px;'>📝 샘플 리뷰</div>"))
        show_sample_reviews(t_df, n=N_SAMPLE)

        # 이미지 군집 분포
        if IMAGE_CLUSTER in t_df.columns:
            img_dist = t_df.groupby(IMAGE_CLUSTER)[IMG_LABEL_COL].first().reset_index()
            img_dist["건수"] = t_df.groupby(IMAGE_CLUSTER).size().values
            display(HTML("<div style='font-size:13px; font-weight:bold; margin:8px 0 4px;'>🖼 이미지 군집 분포</div>"))
            display(img_dist.sort_values("건수", ascending=False))

    # 3. 설치 단계 전체 키워드
    show_header("🔑 설치 단계 전체 상위 키워드", color="#117A65")
    all_words = get_top_words(install_df[TEXT_COL], top_n=TOP_WORDS)
    show_keyword_bar(all_words, title="설치 단계 전체")


# ── 실행 ──────────────────────────────────────────────────────────────────────
analyze_install_stage(df)

,건수
cej_stage,
사용,205
설치/사용/관리,175
구매/설치/사용,78
설치/사용,71


,image_subcluster_global,image_subcluster_visual_label,건수
1,T0_I1,식탁 위 음식·식기 클로즈업,37
0,T0_I0,주방·다이닝 전경 속 냉장고 배치,35


,image_subcluster_global,image_subcluster_visual_label,건수
3,T1_I3,주방·다이닝 통합 전경,31
0,T1_I0,식사·테이블 라이프스타일 장면,28
2,T1_I2,거실·복도 등 비주방 생활공간,18
1,T1_I1,식탁 조명·다이닝 디테일 클로즈업,9


,image_subcluster_global,image_subcluster_visual_label,건수
1,T3_I1,냉장고장 일체형·빌트인 주방 전경,106
0,T3_I0,냉장고 주변 수납·오픈선반 노출형,69


,image_subcluster_global,image_subcluster_visual_label,건수
2,T4_I2,슬림 냉장고 근접 배치,19
1,T4_I1,정면 단독 4도어 냉장고,16
0,T4_I0,냉장고 측면·좁은 공간 배치,9


,image_subcluster_global,image_subcluster_visual_label,건수
0,T7_I0,식탁·조명 중심 다이닝 전경,47


,image_subcluster_global,image_subcluster_visual_label,건수
0,T9_I0,화이트·뉴트럴 아일랜드 주방,24
1,T9_I1,컬러 포인트 감성 주방·다이닝,10


,image_subcluster_global,image_subcluster_visual_label,건수
0,T10_I0,아일랜드 중심 대면형 주방,46
1,T10_I1,거실-다이닝-주방 연결형 공간,25
